[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-11-comet-on-a6.ipynb)


# Lecture 11: COMET on your A6 sentences, against BLEU, chrF and your own ranking

In A6 you scored two MT systems with BLEU and chrF and ranked 50 of their translations by hand.
This adds two learned metrics, COMET with a reference and COMET-QE-DA without one, and asks the
question A11 is about: do the metrics agree with each other, and with you?

By the end you will be able to:

1. score both systems with COMET and COMET-QE-DA on the same sentences as BLEU and chrF;
2. measure how often two metrics, or a metric and you, prefer the same system, sentence by
   sentence;
3. say whether each COMET model agrees with BLEU, chrF and your ranking, and why;
4. say what an LLM-as-judge scores, and why agreeing on the better system is not agreeing on
   each sentence.

These serve Lecture 11's outcome *Run COMET and COMET-QE on their own A6 data; reconcile them
with BLEU, chrF and their human ranking; state what LLM-as-judge scores and its system- against
segment-level caveat*.

You write two functions, marked TODO, and one prediction; checks tell you when each is done, and
the notebook runs the rest. **Running in Google Colab?** Runtime → Change runtime type → any GPU
(a T4 is fine), then Run all; it stops at each TODO until you write it, and at the prediction
until you write one. The scores are saved to Drive as soon as they are computed, so a dropped
session never costs you the scoring: run Step 1 and the cells above it, then Step 3, then
Step 5. **When you
finish, disconnect the GPU**: Runtime → Disconnect and delete runtime. Not while the scoring cell
runs, which would stop it. The last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — homework for Lecture 11 (*Neural Quality Estimation and Evaluation*). A head start on **A11**.

In [ ]:
# Install TorchLingo and COMET in Colab. Nothing to uncomment. Pip's "you may need to restart
# the kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # COMET asks for numpy older than 2, which would break packages Colab ships built for
    # numpy 2; COMET itself runs on numpy 2. So its other requirements go in first, then COMET.
    # pip may then print a red list of conflicts with other packages Colab ships; this uses none.
    %pip install --quiet "entmax>=1.1,<2" "jsonargparse==3.13.1" "pytorch-lightning>=2,<3" "sacrebleu>=2,<3" "setuptools<81" "torchmetrics>=0.10.2,<0.11" "transformers>=4.17,<5"
    %pip install --quiet --no-deps unbabel-comet
    from importlib.metadata import version
    if int(version("numpy").split(".")[0]) < 2:
        raise RuntimeError("An earlier COMET install replaced numpy in this session, which breaks "
                           "COMET. Runtime -> Disconnect and delete runtime, then Run all again.")
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where your A6 files are.
from torchlingo.colab import course_dir, setup

env = setup(drive=True)
A6_DIR = course_dir(env.drive_dir, "assignment6")   # MyDrive/CS479/assignment6


## Step 1: your A6 sentences and your MTEval ranks

Put your A6 files in `MyDrive/CS479/assignment6` as plain text, one sentence per line, line *n*
of each belonging to source sentence *n*: the sources, each system's translations, and the
references. If your A6 data is in another shape, such as one spreadsheet with a column for each,
save each column as its own `.txt` file first. Set the file names below.

Put your **MTEval results file** from A6 there too, and set its name. It has a header row, the
word `source` and then your two systems' names, and one row for each sentence you ranked: the
source, then each system's rank, where 1 is the better of the two and equal ranks are a tie.
**Use the two names in its header as the keys of `SYSTEM_FILES`**, exactly as written there.

The cell after that reads your ranks and finds each ranked sentence in your source file,
ignoring differences in spacing. It says how many it found, lists any it did not, and computes
each system's average rank, the number you reported in A6: a system you preferred every time
averages 1.0, and closer to 1 is better.

In [ ]:
SOURCE_FILE = "source.txt"          # <- your file names, in MyDrive/CS479/assignment6
REFERENCE_FILE = "reference.txt"
SYSTEM_FILES = {"system 1": "system1.txt", "system 2": "system2.txt"}   # <- your MTEval header's names
MTEVAL_FILE = "mteval_results.tsv"  # <- your MTEval results file from A6

# Stops, naming every file it cannot find, before reading any of them.
course_dir(
    env.drive_dir, "assignment6",
    needs=[SOURCE_FILE, REFERENCE_FILE, *SYSTEM_FILES.values(), MTEVAL_FILE],
    hint="Put your A6 files there, or fix the names above.",
)

In [ ]:
# Setup: run this cell, no need to read it. It reads your A6 files and your MTEval ranks.


def read_lines(name):
    """The file's lines. utf-8-sig drops a byte-order mark; reading as text turns Windows line
    endings into "\n"; splitting on "\n" alone keeps other line-break characters inside a line."""
    return (A6_DIR / name).read_text(encoding="utf-8-sig").rstrip("\n").split("\n")


def same_text(sentence):
    """The sentence with its spacing normalized, to match MTEval's rows to your sources."""
    return " ".join(sentence.split())


def read_mteval(name, systems):
    """Return the ranked sources, and {system: [rank, ...]}, from MTEval's results file."""
    lines = read_lines(name)
    header = lines[0].split("\t")
    if len(header) != 3 or set(header[1:]) != set(systems):
        raise SystemExit(
            f"{name}'s header is {header}: a source column, then your two systems' names. "
            f"SYSTEM_FILES names {list(systems)}; use the header's two names as its keys."
        )
    ranked, ranks = [], {system: [] for system in header[1:]}
    for n, line in enumerate(lines[1:], start=2):
        cells = line.split("\t")
        if len(cells) != 3 or not all(cell.strip().isdigit() for cell in cells[1:]):
            raise SystemExit(f"{name}, line {n}: expected a sentence and two ranks, got {cells}")
        ranked.append(cells[0])
        for system, cell in zip(header[1:], cells[1:]):
            ranks[system].append(int(cell))
    return ranked, ranks


NOTES = A6_DIR / "a11-notes.json"   # the scoring cell saves the scores here; the report reads them
sources, references = read_lines(SOURCE_FILE), read_lines(REFERENCE_FILE)
translations = {name: read_lines(f) for name, f in SYSTEM_FILES.items()}
print(f"{len(sources):,} source sentences; systems: {', '.join(SYSTEM_FILES)}.")

ranked, ranks = read_mteval(MTEVAL_FILE, SYSTEM_FILES)
line_of = {}
for i, sentence in enumerate(sources):
    line_of.setdefault(same_text(sentence), i)
matched = [line_of.get(same_text(sentence)) for sentence in ranked]   # a line of your files, or None
found = sum(i is not None for i in matched)
print(f"{MTEVAL_FILE}: {len(ranked)} sentences you ranked; {found} found in {SOURCE_FILE}.")
for sentence, i in zip(ranked, matched):
    if i is None:
        print(f"  not found: {sentence}")
if not found:
    raise SystemExit(f"None of the sentences you ranked is in {SOURCE_FILE}. Is it the source "
                     "file you gave MTEval?")

HUMAN_AVERAGE_RANK = {system: sum(r) / len(r) for system, r in ranks.items()}
ties = sum(r1 == r2 for r1, r2 in zip(*ranks.values()))
print("Your average ranks, as in A6: "
      + ", ".join(f"{system} {rank:.2f}" for system, rank in HUMAN_AVERAGE_RANK.items())
      + f"; {ties} of {len(ranked)} ranked as a tie.")

## Step 2: `comet_inputs`, from A10

**Yours: the function you wrote in A10**, which turns aligned sentences into the records COMET
scores and refuses lists of different lengths. Paste it in place of the TODO; the same checks
run.


In [ ]:
def comet_inputs(sources, translations, references=None):
    """Return one COMET record per sentence: src, mt, and ref when references are given.

    TODO: paste your A10 function. It raises ValueError if translations, or references when
    given, are not the same length as sources.
    """
    raise NotImplementedError("paste comet_inputs from A10, then run this cell again")


got = comet_inputs(["hola"], ["hello"], ["hi"])
assert got == [{"src": "hola", "mt": "hello", "ref": "hi"}], (
    f"with a reference, each record is {{'src': ..., 'mt': ..., 'ref': ...}}; got {got}")
got = comet_inputs(["hola"], ["hello"])
assert got == [{"src": "hola", "mt": "hello"}], (
    f"without a reference, each record has only 'src' and 'mt'; got {got}")
for bad in ((["a", "b"], ["x"], None), (["a"], ["x"], ["r", "s"])):
    try:
        got = comet_inputs(*bad)
    except ValueError:
        continue
    raise AssertionError(f"comet_inputs must raise ValueError for lists of different lengths, "
                         f"{bad}; got {got}")
for name, mt in translations.items():
    comet_inputs(sources, mt, references)   # your files are aligned, or this says which is not
print("comet_inputs passes its checks, and your files are aligned.")

## Step 3: yours to write, `pairwise_agreement`

With two systems, "do two metrics correlate?" has a sentence-by-sentence answer: for each
sentence, does each metric prefer the same system? Step 5 also uses it on your MTEval ranks,
which is how it measures how often a metric agrees with you.

`pairwise_agreement(a_1, a_2, b_1, b_2)` takes two metrics' scores, metric a's for systems 1 and
2 and metric b's for the same two, one score per sentence, and returns `(agree, counted)`:

- Higher is better for every score here. For sentence *i*, metric a prefers system 1 if
  `a_1[i] > a_2[i]`, system 2 if `a_1[i] < a_2[i]`, and **neither if they are equal**: a tie is
  no preference. Likewise metric b.
- `counted` is the number of sentences where **both** metrics prefer a system.
- `agree` is how many of those they prefer the same one on.

In [ ]:
def pairwise_agreement(a_1, a_2, b_1, b_2):
    """Return (agree, counted) for metric a's and metric b's scores of systems 1 and 2.

    TODO: write it, as Step 3 above specifies.
    """
    raise NotImplementedError("write pairwise_agreement, then run this cell again")


# A case you can check by hand: agree, disagree, a tie in a, a tie in b, agree.
got = pairwise_agreement([2, 1, 5, 3, 0.9], [1, 2, 5, 1, 0.1],
                         [9, 9, 1, 4, 0.7], [1, 1, 2, 4, 0.2])
assert got == (2, 3), (
    f"pairwise_agreement is not right yet: here it should count 2 agreements out of 3, (2, 3); "
    f"got {got}")
print("pairwise_agreement passes its check.")

## Step 4: predict, then score

**Write a prediction** in `PREDICTION`: will COMET prefer the same system as BLEU, chrF and your
ranking, and will COMET-QE-DA agree with COMET? The cell stops until you write one: a prediction
made after seeing the scores teaches nothing.


In [ ]:
PREDICTION = ""   # <- your prediction, in a sentence or two

assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."
print("Prediction recorded.")


Now the next cell scores both systems with all four metrics, on every sentence. On a GPU this
takes a few minutes for 500 sentences. **Leave the runtime connected while it runs**:
disconnecting stops it, and it starts again from the beginning. When it ends, it saves the scores
to `a11-notes.json` beside your A6 files on Drive.

**Why COMET-QE-DA, not A10's COMETKiwi?** Both score a translation from the source alone, with no
reference: that is estimation, where COMET with a reference is evaluation. COMETKiwi is the newer
of the two, but it downloads only with a token, for an account that has accepted its license;
COMET-QE-DA needs neither, so this notebook runs for everyone. If COMETKiwi worked for you in
A10, you may swap in `Unbabel/wmt22-cometkiwi-da` after logging in as you did there; the report
still labels it COMET-QE-DA, so say in your summary that you did.

In [ ]:
import json

import torch
from comet import download_model, load_from_checkpoint
from sacrebleu.metrics import CHRF
from torchlingo.evaluation import compute_bleu, compute_chrf

GPUS = 1 if torch.cuda.is_available() else 0
sentence_chrf = CHRF(word_order=2)          # the chrF++ that compute_chrf uses

# Each metric's score for each system, overall and per sentence: system_scores["COMET"]["DeepL"].
system_scores, sentence_scores = {}, {}
for model_name, label, with_refs in (("Unbabel/wmt22-comet-da", "COMET", True),
                                     ("Unbabel/wmt20-comet-qe-da", "COMET-QE-DA", False)):
    model = load_from_checkpoint(download_model(model_name))
    system_scores[label], sentence_scores[label] = {}, {}
    for name, mt in translations.items():
        out = model.predict(comet_inputs(sources, mt, references if with_refs else None),
                            batch_size=32, gpus=GPUS, num_workers=2, progress_bar=False)
        system_scores[label][name] = float(out.system_score)
        sentence_scores[label][name] = [float(s) for s in out.scores]

system_scores["BLEU"], system_scores["chrF"], sentence_scores["chrF"] = {}, {}, {}
for name, mt in translations.items():
    system_scores["BLEU"][name] = compute_bleu(mt, references).score
    system_scores["chrF"][name] = compute_chrf(mt, references).score
    sentence_scores["chrF"][name] = [sentence_chrf.sentence_score(h, [r]).score
                                     for h, r in zip(mt, references)]

# Saved to Drive, so the report survives a dropped session without scoring again.
NOTES.write_text(json.dumps({"prediction": PREDICTION.strip(), "system_scores": system_scores,
                             "sentence_scores": sentence_scores}), encoding="utf-8")
print(f"Scored with COMET, COMET-QE-DA, BLEU and chrF; saved to {NOTES.name} on Drive.")

## Step 5: the report

The system scores, which system each metric prefers, and, sentence by sentence, how often the
metrics agree with each other and with you. With two systems, "correlates with your ranking" at
the system level means "prefers the same system"; the sentence-level counts say how firmly.
Against you, the counts use your own `pairwise_agreement` on the sentences you ranked, so a
sentence you ranked as a tie expresses no preference and is left out.

The report reads the scores from `a11-notes.json` on Drive. If Colab dropped your session after
scoring, you need not score again: run Step 1 and the cells above it, then Step 3, then this
cell.

In [ ]:
import json

notes = json.loads(NOTES.read_text(encoding="utf-8"))   # saved by the scoring cell
system_scores, sentence_scores = notes["system_scores"], notes["sentence_scores"]

one, two = SYSTEM_FILES
print("=== For the A11 summary ===")
print(f"{len(sources):,} sentences scored automatically; systems: {one} and {two}\n")
print(f"{'metric':12} {one:>12} {two:>12}   prefers")
for label in ("BLEU", "chrF", "COMET", "COMET-QE-DA"):
    s1, s2 = system_scores[label][one], system_scores[label][two]
    print(f"{label:12} {s1:12.3f} {s2:12.3f}   {one if s1 > s2 else two}")
h1, h2 = HUMAN_AVERAGE_RANK[one], HUMAN_AVERAGE_RANK[two]
prefers = one if h1 < h2 else two if h2 < h1 else "neither"
print(f"{'human rank':12} {h1:12.2f} {h2:12.2f}   {prefers}  (lower is better; all {len(ranked)} you ranked)")

print("\nSentence by sentence, how often two metrics prefer the same system:")
for a, b in (("COMET", "chrF"), ("COMET-QE-DA", "chrF"), ("COMET-QE-DA", "COMET")):
    agree, counted = pairwise_agreement(sentence_scores[a][one], sentence_scores[a][two],
                                        sentence_scores[b][one], sentence_scores[b][two])
    print(f"  {a} and {b}: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")

# Your ranks as scores where higher is better, like every score above: minus the rank. A
# sentence you ranked as a tie is then no preference, and pairwise_agreement leaves it out.
rows = [(k, i) for k, i in enumerate(matched) if i is not None]
you_1, you_2 = [-ranks[one][k] for k, _ in rows], [-ranks[two][k] for k, _ in rows]
ties = sum(r1 == r2 for r1, r2 in zip(you_1, you_2))
print(f"\nOn the {len(rows)} sentences you ranked and Step 1 found, how often each metric prefers "
      f"the system you did; {ties} you ranked as a tie are not counted:")
for label in ("chrF", "COMET", "COMET-QE-DA"):
    agree, counted = pairwise_agreement([sentence_scores[label][one][i] for _, i in rows],
                                        [sentence_scores[label][two][i] for _, i in rows],
                                        you_1, you_2)
    print(f"  {label} and you: {agree} of {counted} ({100 * agree / max(counted, 1):.0f}%)")
print(f"\nYour prediction: {notes['prediction']}")

## Step 6: the summary

A11's one-page summary reports the numbers above and answers these, each in a few sentences:

1. **Does each COMET model agree with BLEU, chrF and your ranking, and why or why not?** Judge
   the reference-based COMET and COMET-QE-DA separately: their scores sit on different scales, so
   compare which system each prefers, never one model's number with the other's. Answer for the
   system each prefers, and for the sentences you ranked, one by one.
2. **Your prediction against the result.** Where were you right, where wrong, and what does the
   miss tell you?
3. **What the reference changes.** COMET and COMET-QE-DA read the same translations; one also
   reads your reference. Find a sentence where they disagree and say what the reference told
   COMET that the source alone did not.
4. **LLM-as-judge.** Suppose you asked a large language model to score each translation from 0 to
   100, given the source. Is that evaluation or estimation? Such judges agree with people far
   better on which *system* is better than on which *sentence* is: use your own Step 5 counts,
   where a metric can prefer the same system you did while disagreeing with you on many of the
   sentences you ranked, to explain how that can be.

For the extra credit, xCOMET-XL is `Unbabel/XCOMET-XL` in the same loop. Like COMETKiwi in A10,
it needs its license accepted on Hugging Face and a token, and it needs a large GPU.

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Your scores are saved on Drive in `a11-notes.json`, beside your A6 files, so you can print the report again later without scoring again: run Step 1 and the cells above it, then Step 3, then Step 5.